"""
Ex-post volatility shock detection on S&P 500 daily data.

Reads the feature set already produced by snp.ipynb (sp500_daily.csv) and
detects abnormal-volatility "shock" days using four methods, ordered from
simplest to most robust, per Family 1 of the pre-analysis plan:

    1. Rolling realized-vol threshold breach   (simplest, fully transparent)
    2. Drawdown-based                          (secondary screen, captures
                                                 sharp down-moves rolling-vol
                                                 measures can be slow to flag)
    3. GARCH(1,1) / GJR-GARCH conditional vol   (adapts to vol clustering,
                                                 won't over-flag already-
                                                 turbulent regimes)
    4. Jump detection (Lee-Mykland style)       (most robust: distinguishes
                                                 genuine discontinuities from
                                                 ordinary high-vol diffusion)

Each method outputs:
    - a boolean "is_shock" column appended to the daily frame
    - an event table (one row per contiguous shock run) with onset, offset,
      peak day, peak magnitude, and the threshold/parameters used

Design choices are deliberately explicit and adjustable at the top of each
function so threshold sensitivity (k = 2, 2.5, 3, ...) can be swept and
documented, per the pre-analysis plan's robustness-check requirement.

Outputs:
    shocks_rolling_vol.csv
    shocks_drawdown.csv
    shocks_garch.csv
    shocks_jump.csv
    shocks_combined_summary.csv   (all four side-by-side, for triangulation)
"""

In [2]:
import os
import numpy as np
import pandas as pd
from scipy import stats

try:
    from arch import arch_model
    HAVE_ARCH = True
except ImportError:
    HAVE_ARCH = False


# ==================================================================
# I/O
# ==================================================================

INPUT_PATH = "C:/Python/CSUREMM/data_primitive/sp500_daily.csv"   # output of snp.ipynb
OUTPUT_DIR = "C:/Python/CSUREMM/shock_detection"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_sp500(path=INPUT_PATH):
    df = pd.read_csv(path, index_col="date", parse_dates=True)
    df = df.sort_index()
    return df


# ==================================================================
# Shared helper: turn a boolean flag series into contiguous event runs
# ==================================================================

def flags_to_events(df, flag_col, magnitude_col, method_name, params):
    """
    Collapse a day-level boolean flag column into one row per contiguous
    run of flagged days (an "event"), with onset/offset/peak info.

    This is the same logic regardless of which detector produced the flag,
    so all four methods produce directly comparable event tables.
    """
    flags = df[flag_col].fillna(False).astype(bool)
    if not flags.any():
        return pd.DataFrame(columns=[
            "method", "onset", "offset", "duration_days",
            "peak_date", "peak_value", "params"
        ])

    # group consecutive True runs
    run_id = (flags != flags.shift(fill_value=False)).cumsum()
    events = []
    for rid, grp_idx in flags[flags].groupby(run_id[flags]).groups.items():
        idx = list(grp_idx)
        onset, offset = idx[0], idx[-1]
        seg = df.loc[onset:offset, magnitude_col]
        peak_date = seg.idxmax() if seg.notna().any() else onset
        peak_value = seg.max() if seg.notna().any() else np.nan
        events.append({
            "method": method_name,
            "onset": onset,
            "offset": offset,
            "duration_days": (idx.index(offset) - idx.index(onset) + 1)
                              if False else len(idx),
            "peak_date": peak_date,
            "peak_value": peak_value,
            "params": str(params),
        })
    return pd.DataFrame(events).sort_values("onset").reset_index(drop=True)


# ==================================================================
# METHOD 1 (easiest): Rolling realized-vol threshold breach
# ==================================================================
#
# Flags a day as a shock day if a realized-vol measure exceeds its own
# trailing baseline by k standard deviations. Uses the rv5/rv10/rv21
# columns already computed in snp.ipynb -- no new vol estimator needed.
#
# Pros: fully transparent, one line of logic, easy to explain in a methods
#       section, easy to sensitivity-sweep over k.
# Cons: trailing baseline is backward-looking only (no clustering model),
#       a long-trailing window can be slow to "forget" a past shock, which
#       can bias the baseline upward right after a shock and suppress
#       detection of a second nearby shock.

def detect_rolling_vol_shock(
    df,
    vol_col="sp500_rv5",
    baseline_window=60,
    k=2.5,
    min_periods=30,
):
    out = df.copy()
    vol = out[vol_col]

    baseline_mean = vol.rolling(baseline_window, min_periods=min_periods).mean()
    baseline_std = vol.rolling(baseline_window, min_periods=min_periods).std()

    # shift baseline by 1 so today's vol isn't used to define today's own
    # threshold (avoids look-ahead / self-referential flagging)
    baseline_mean = baseline_mean.shift(1)
    baseline_std = baseline_std.shift(1)

    out["rolling_vol_threshold"] = baseline_mean + k * baseline_std
    out["rolling_vol_zscore"] = (vol - baseline_mean) / baseline_std
    out["is_shock_rolling_vol"] = vol > out["rolling_vol_threshold"]

    params = {"vol_col": vol_col, "baseline_window": baseline_window, "k": k}
    events = flags_to_events(out, "is_shock_rolling_vol", vol_col,
                              "rolling_vol", params)
    return out, events


# ==================================================================
# METHOD 2: Drawdown-based
# ==================================================================
#
# Flags the *onset* of a cumulative drawdown exceeding a threshold within
# a short trailing window (e.g. -5% over 3 trading days). Catches sharp
# directional moves that a vol estimator can under-weight if the move is
# a single huge down day surrounded by otherwise-calm days (rv5/rv10 are
# averages, so one big day gets diluted).
#
# NOTE: this is direction-asymmetric by construction (flags drawdowns,
# not draw-ups). Per the pre-analysis plan, run a mirrored "run-up" version
# too if upside shocks (squeezes, surprise dovish pivots) matter to you --
# the function below does both and tags the direction.

def detect_drawdown_shock(
    df,
    price_col="sp500_close",
    window=3,
    threshold=-0.05,     # -5% over `window` trading days = down-shock
    upside_threshold=0.05,  # +5% over `window` trading days = up-shock
):
    out = df.copy()
    price = out[price_col]

    roll_change = price.pct_change(window)
    out["drawdown_pct"] = roll_change

    out["is_shock_drawdown_down"] = roll_change <= threshold
    out["is_shock_drawdown_up"] = roll_change >= upside_threshold
    out["is_shock_drawdown"] = (
        out["is_shock_drawdown_down"] | out["is_shock_drawdown_up"]
    )

    # magnitude column for peak-finding: absolute drawdown size
    out["drawdown_abs"] = roll_change.abs()

    params = {"price_col": price_col, "window": window,
              "down_threshold": threshold, "up_threshold": upside_threshold}
    events = flags_to_events(out, "is_shock_drawdown", "drawdown_abs",
                              "drawdown", params)

    # tag direction on the event table by checking the sign at peak_date
    if len(events):
        events["direction"] = events["peak_date"].map(
            lambda d: "down" if out.loc[d, "drawdown_pct"] < 0 else "up"
        )
    return out, events


# ==================================================================
# METHOD 3: GARCH(1,1) / GJR-GARCH conditional volatility
# ==================================================================
#
# Fits a conditional-vol model so the "baseline" adapts to vol clustering
# instead of being a flat trailing window. Flags a day as a shock if the
# *standardized residual* (return / conditional vol) is extreme, OR if the
# conditional vol itself jumps sharply day-over-day (a vol *innovation*,
# distinct from just "vol happens to be high").
#
# GJR-GARCH variant captures the well-documented leverage effect (bad news
# raises vol more than equally-sized good news) -- worth using as the
# primary spec rather than plain GARCH(1,1) given markets' known asymmetry.
#
# Pros: model-based, adapts to clustering, distinguishes "vol is high
#       because we're already in a turbulent regime" from "today's return
#       is unusual EVEN GIVEN the turbulent regime."
# Cons: requires choosing a model, harder to explain to a non-technical
#       reader, sensitive to estimation window / refit frequency.

def detect_garch_shock(
    df,
    return_col="sp500_log_return_1d",
    dist="t",               # Student-t handles fat tails better than normal
    z_threshold=2.5,         # standardized-residual threshold
    vol_innovation_k=2.0,    # threshold for day-over-day jump in cond. vol
    refit_every=None,        # None = single full-sample fit (simplest);
                              # set an int (e.g. 252) for rolling refits
):
    if not HAVE_ARCH:
        raise ImportError("pip install arch --break-system-packages")

    out = df.copy()
    rets = out[return_col].dropna() * 100  # arch package expects pct units

    am = arch_model(
        rets, vol="GARCH", p=1, o=1, q=1,  # o=1 -> GJR-GARCH asymmetry term
        dist=dist, rescale=False,
    )
    res = am.fit(disp="off")

    cond_vol = res.conditional_volatility / 100  # back to return units
    std_resid = res.std_resid  # (return - mean) / conditional_vol, dimensionless

    out.loc[rets.index, "garch_cond_vol"] = cond_vol
    out.loc[rets.index, "garch_std_resid"] = std_resid

    # vol innovation: how much today's conditional vol jumped vs trailing
    # mean of conditional vol (catches the *onset* of a new high-vol regime)
    cv = out["garch_cond_vol"]
    cv_baseline = cv.rolling(60, min_periods=30).mean().shift(1)
    cv_baseline_std = cv.rolling(60, min_periods=30).std().shift(1)
    out["garch_vol_innovation_z"] = (cv - cv_baseline) / cv_baseline_std

    out["is_shock_garch_residual"] = out["garch_std_resid"].abs() > z_threshold
    out["is_shock_garch_vol_jump"] = out["garch_vol_innovation_z"] > vol_innovation_k
    out["is_shock_garch"] = (
        out["is_shock_garch_residual"] | out["is_shock_garch_vol_jump"]
    )

    params = {"dist": dist, "z_threshold": z_threshold,
              "vol_innovation_k": vol_innovation_k, "model": "GJR-GARCH(1,1,1)"}
    events = flags_to_events(out, "is_shock_garch", "garch_cond_vol",
                              "garch", params)
    return out, events, res


# ==================================================================
# METHOD 4 (most robust): Jump detection (Lee-Mykland style)
# ==================================================================
#
# Distinguishes a genuine price *discontinuity* (a "jump") from an unusual
# but continuous diffusive move. Uses a bipower-variation estimate of
# local continuous volatility (robust to jumps by construction -- the
# estimator itself is designed to not be contaminated by the jumps it's
# trying to detect), then flags a day where the standardized return is
# extreme relative to that jump-robust local vol estimate.
#
# This is the right tool when you specifically want to claim "something
# discontinuous happened here" (a true shock) rather than "vol is locally
# elevated" (which a long stretch of merely turbulent-but-continuous
# trading would also trigger in Methods 1-3).
#
# Reference: Lee, S. & Mykland, P. (2008), "Jumps in Financial Markets:
# A New Nonparametric Test and Jump Dynamics," Review of Financial Studies.

def detect_jump_shock(
    df,
    return_col="sp500_log_return_1d",
    window=20,          # local window K for bipower variation
    z_threshold=None,   # if None, use the LM-paper asymptotic critical value
    significance=0.01,  # used to build the asymptotic critical value if
                          # z_threshold is None
):
    out = df.copy()
    r = out[return_col]

    # Bipower variation: sum of |r_t| * |r_{t-1}| over the window, robust
    # to a single large jump because consecutive-product terms involving
    # the jump are down-weighted relative to squared-return estimators.
    abs_r = r.abs()
    bpv = (abs_r * abs_r.shift(1)).rolling(window).sum() * (np.pi / 2) / (window - 1)
    local_vol = np.sqrt(bpv)

    # Lee-Mykland statistic: standardized return using the LAGGED local vol
    # (shift so today's own return can't contaminate its own denominator)
    local_vol_lagged = local_vol.shift(1)
    lm_stat = r / local_vol_lagged

    out["jump_local_vol"] = local_vol_lagged
    out["jump_lm_stat"] = lm_stat

    if z_threshold is None:
        # Asymptotic critical value from Lee & Mykland (2008): a Gumbel-type
        # extreme-value threshold that depends on sample size n and the
        # chosen significance level. Using n = effective sample size of
        # the series (non-NaN local_vol observations).
        n_eff = local_vol_lagged.notna().sum()
        c = np.sqrt(2 * np.log(n_eff)) if n_eff > 1 else 3.0
        c_corr = (
            c
            - (np.log(np.pi) + np.log(np.log(n_eff))) / (2 * c)
            if n_eff > 1 else c
        )
        beta_star = -np.log(-np.log(1 - significance))
        z_threshold = c_corr + beta_star / c_corr
        z_threshold = float(np.clip(z_threshold, 3.0, 8.0))  # sanity bounds

    out["is_shock_jump"] = lm_stat.abs() > z_threshold

    params = {"window": window, "z_threshold": round(float(z_threshold), 3),
              "significance": significance, "estimator": "Lee-Mykland bipower"}
    events = flags_to_events(out, "is_shock_jump", "jump_lm_stat",
                              "jump", params)
    # use abs value for peak magnitude comparability
    if len(events):
        events["peak_value"] = events["peak_value"].abs()
    return out, events


# ==================================================================
# Combine all four for triangulation
# ==================================================================

def combine_methods(events_list):
    """
    Stack all method event tables and, separately, build a day-level
    "how many methods agree" tally -- useful for the triangulation step
    in the pre-analysis plan (events flagged by more methods are more
    likely to be genuine shocks rather than a single estimator's artifact).
    """
    combined = pd.concat(events_list, ignore_index=True)
    return combined.sort_values(["onset", "method"]).reset_index(drop=True)


def agreement_table(df, flag_cols):
    """
    Day-level table counting how many of the four detectors fired on each
    date. flag_cols: list of the is_shock_* column names to compare.
    """
    present = [c for c in flag_cols if c in df.columns]
    tally = df[present].fillna(False).astype(bool).sum(axis=1)
    out = pd.DataFrame({"n_methods_agree": tally})
    for c in present:
        out[c] = df[c].fillna(False).astype(bool)
    return out[out["n_methods_agree"] > 0].sort_index()


# ==================================================================
# Main
# ==================================================================

def main():
    df = load_sp500()
    print(f"[load] {df.shape[0]} rows, {df.index.min().date()} to {df.index.max().date()}")

    all_events = []

    # ---- Method 1: rolling vol threshold ----
    df, ev1 = detect_rolling_vol_shock(df, vol_col="sp500_rv5",
                                        baseline_window=60, k=2.5)
    print(f"[method 1: rolling vol] {len(ev1)} events")
    ev1.to_csv(f"{OUTPUT_DIR}/shocks_rolling_vol.csv", index=False)
    all_events.append(ev1)

    # ---- Method 2: drawdown ----
    df, ev2 = detect_drawdown_shock(df, price_col="sp500_close",
                                     window=3, threshold=-0.05,
                                     upside_threshold=0.05)
    print(f"[method 2: drawdown] {len(ev2)} events")
    ev2.to_csv(f"{OUTPUT_DIR}/shocks_drawdown.csv", index=False)
    all_events.append(ev2)

    # ---- Method 3: GARCH ----
    if HAVE_ARCH:
        df, ev3, garch_res = detect_garch_shock(
            df, return_col="sp500_log_return_1d",
            dist="t", z_threshold=2.5, vol_innovation_k=2.0,
        )
        print(f"[method 3: GJR-GARCH] {len(ev3)} events")
        ev3.to_csv(f"{OUTPUT_DIR}/shocks_garch.csv", index=False)
        all_events.append(ev3)
    else:
        print("[method 3: GJR-GARCH] skipped -- `arch` package not installed")

    # ---- Method 4: jump detection ----
    df, ev4 = detect_jump_shock(df, return_col="sp500_log_return_1d",
                                 window=20, significance=0.01)
    print(f"[method 4: jump detection] {len(ev4)} events")
    ev4.to_csv(f"{OUTPUT_DIR}/shocks_jump.csv", index=False)
    all_events.append(ev4)

    # ---- combined ----
    combined = combine_methods(all_events)
    combined.to_csv(f"{OUTPUT_DIR}/shocks_combined_summary.csv", index=False)

    flag_cols = ["is_shock_rolling_vol", "is_shock_drawdown",
                 "is_shock_garch", "is_shock_jump"]
    agree = agreement_table(df, flag_cols)
    agree.to_csv(f"{OUTPUT_DIR}/shocks_agreement_by_day.csv")
    print(f"[agreement] {len(agree)} days flagged by >=1 method; "
          f"{(agree['n_methods_agree'] >= 3).sum()} days flagged by >=3 methods")

    # full annotated daily frame, for inspection / plotting later
    df.to_csv(f"{OUTPUT_DIR}/sp500_daily_with_shock_flags.csv")
    print(f"[save] annotated daily frame -> sp500_daily_with_shock_flags.csv")


if __name__ == "__main__":
    main()

[load] 1105 rows, 2022-01-03 to 2026-05-29
[method 1: rolling vol] 13 events
[method 2: drawdown] 15 events
[method 3: GJR-GARCH] 42 events
[method 4: jump detection] 3 events
[agreement] 120 days flagged by >=1 method; 8 days flagged by >=3 methods
[save] annotated daily frame -> sp500_daily_with_shock_flags.csv
